# Root Listener Tracers

The `root_listeners.py` module defines synchronous and asynchronous tracers that call listener functions for the root run of a Runnable execution.

Each tracer invokes an optional start listener when the first run is created. When that root run is updated, it invokes either the completion listener or error listener based on whether the run contains an error. Child-run lifecycle events are ignored.

## Type Aliases

1. `Listener`: Represents a synchronous root-run listener.

   A listener may accept only the `Run` or accept both the `Run` and its `RunnableConfig`.

   * **Definition:**
     ```python
     Listener = (
         Callable[
             [Run],
             None
         ]
         | Callable[
             [Run, RunnableConfig],
             None
         ]
     )
     ```

2. `AsyncListener`: Represents an asynchronous root-run listener.

   A listener may accept only the `Run` or accept both the `Run` and its `RunnableConfig`.

   * **Definition:**
     ```python
     AsyncListener = (
         Callable[
             [Run],
             Awaitable[None]
         ]
         | Callable[
             [Run, RunnableConfig],
             Awaitable[None]
         ]
     )
     ```

# RootListenersTracer

`RootListenersTracer` is a synchronous tracer that calls listener functions for the start, successful completion, and failed completion of the root run.

The first run observed by the tracer becomes its root run. Later child runs do not trigger the configured listeners.

## Bases

- `BaseTracer`

## Attributes

1. `log_missing_parent`: Disables warnings or debug logging when a parent run is missing.
   * **Type:**
     ```python
     log_missing_parent: bool = False
     ```

2. `config`: Stores the Runnable configuration passed to listener functions that accept a configuration argument.
   * **Type:**
     ```python
     config: RunnableConfig
     ```

3. `root_id`: Stores the identifier of the first run observed by the tracer.

   It remains `None` until the first run is created.

   * **Type:**
     ```python
     root_id: UUID | None
     ```

### Methods

1. `__init__`: Creates a synchronous root-listener tracer.

   The tracer uses the `"original+chat"` internal schema format so chat-model runs can be handled directly. Listener functions are optional.

   * **Syntax:**
     ```python
     __init__(
         self,
         *,
         config: RunnableConfig, # Runnable configuration supplied to listeners
         on_start: Listener | None, # Listener called when the root run starts
         on_end: Listener | None, # Listener called when the root run succeeds
         on_error: Listener | None # Listener called when the root run fails
     ) -> None
     ```

2. `_persist_run`: Implements the tracer persistence contract without performing an action.

   Root listener behaviour is handled through `_on_run_create` and `_on_run_update`. The legacy persistence hook is not useful because it is called only once for the complete run tree.

   * **Syntax:**
     ```python
     _persist_run(
         self,
         run: Run # Completed root run
     ) -> None
     ```

3. `_on_run_create`: Records the first run as the root run and calls the configured start listener.

   Later runs are ignored. The listener is invoked through `call_func_with_variable_args`, which supplies the Runnable configuration only when the listener accepts it.

   * **Syntax:**
     ```python
     _on_run_create(
         self,
         run: Run # Newly created run
     ) -> None
     ```

4. `_on_run_update`: Calls the configured completion or error listener when the root run is updated.

   Child-run updates are ignored. When `run.error` is `None`, `on_end` is called. Otherwise, `on_error` is called.

   * **Syntax:**
     ```python
     _on_run_update(
         self,
         run: Run # Updated run
     ) -> None
     ```

# AsyncRootListenersTracer

`AsyncRootListenersTracer` is an asynchronous tracer that awaits listener functions for the start, successful completion, and failed completion of the root run.

The first run observed by the tracer becomes its root run. Later child runs do not trigger the configured listeners.

## Bases

- `AsyncBaseTracer`

## Attributes

1. `log_missing_parent`: Disables warnings or debug logging when a parent run is missing.
   * **Type:**
     ```python
     log_missing_parent: bool = False
     ```

2. `config`: Stores the Runnable configuration passed to listener functions that accept a configuration argument.
   * **Type:**
     ```python
     config: RunnableConfig
     ```

3. `root_id`: Stores the identifier of the first run observed by the tracer.

   It remains `None` until the first run is created.

   * **Type:**
     ```python
     root_id: UUID | None
     ```

### Methods

1. `__init__`: Creates an asynchronous root-listener tracer.

   The tracer uses the `"original+chat"` internal schema format so chat-model runs can be handled directly. Listener functions are optional.

   * **Syntax:**
     ```python
     __init__(
         self,
         *,
         config: RunnableConfig, # Runnable configuration supplied to listeners
         on_start: AsyncListener | None, # Listener awaited when the root run starts
         on_end: AsyncListener | None, # Listener awaited when the root run succeeds
         on_error: AsyncListener | None # Listener awaited when the root run fails
     ) -> None
     ```

2. `_persist_run`: Asynchronously implements the tracer persistence contract without performing an action.

   Root listener behaviour is handled through `_on_run_create` and `_on_run_update`. The legacy persistence hook is not useful because it is called only once for the complete run tree.

   * **Syntax:**
     ```python
     async _persist_run(
         self,
         run: Run # Completed root run
     ) -> None
     ```

3. `_on_run_create`: Records the first run as the root run and awaits the configured start listener.

   Later runs are ignored. The listener is invoked through `acall_func_with_variable_args`, which supplies the Runnable configuration only when the listener accepts it.

   * **Syntax:**
     ```python
     async _on_run_create(
         self,
         run: Run # Newly created run
     ) -> None
     ```

4. `_on_run_update`: Awaits the configured completion or error listener when the root run is updated.

   Child-run updates are ignored. When `run.error` is `None`, `on_end` is awaited. Otherwise, `on_error` is awaited.

   * **Syntax:**
     ```python
     async _on_run_update(
         self,
         run: Run # Updated run
     ) -> None
     ```

## Listener Invocation Behaviour

The helper functions used by these tracers inspect each listener's accepted parameters:

- A one-argument listener receives only the root `Run`.
- A two-argument listener receives the root `Run` and the stored `RunnableConfig`.

Only the first run observed by a tracer is treated as the root run. All child-run creation and update events are ignored by the listener layer.